# Three-dimensional visualization of CDU selection

Each image is represented by its mean uncertainty for `crack`, `patch`, and `pothole`. Three-dimensional visualization is possible because the study has exactly three classes, with one uncertainty component on each axis. The figures below are interactive: they can be rotated, zoomed, and inspected with the pointer. Each stage occupies a separate cell so the construction of the acquisition batch remains explicit.


## Profiles and reproduced calculation

`notebooks/data/uncertainty_profiles.pkl` stores the uncertainty vectors for the reference iteration. When this notebook runs, groups and selections are recomputed by the same functions used by `CDUFM4SLOpt055`:

1. `optimize_n_clusters` selects the GMM with the lowest Davies–Bouldin index between $K$ and $5K$ groups;
2. `allocate_candidate_quotas` distributes $4 \times 20=80$ candidates with Sainte-Laguë;
3. `refine_cluster_candidates` applies $\mathrm{score}(i,C_g)=\alpha U(i)+(1-\alpha)\mathrm{diversity}(i,C_g)$ with $\alpha=0.55$;
4. `retain_final_batch` retains the 20 highest uncertainty values $U(i)$ among the candidates.

The reference to round 36 corresponds to 720 pool acquisitions ($36 \times 20$).


In [ ]:
from pathlib import Path
import sys

import pandas as pd
import plotly.io as pio
from IPython.display import display

CANDIDATE_ROOTS = [Path.cwd(), Path.cwd().parent]
ROOT = next((path.resolve() for path in CANDIDATE_ROOTS
             if (path / 'run.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Run the notebook from repo/ or repo/notebooks/.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
pio.renderers.default = 'plotly_mimetype'

from active_learning_benchmark.cluster_visualization import (
    build_cluster_snapshot_visualizations,
)

SNAPSHOT = ROOT / 'notebooks' / 'data' / 'uncertainty_profiles.pkl'
OUTPUT = ROOT / 'figures' / 'clustering'
artifacts = build_cluster_snapshot_visualizations(SNAPSHOT, OUTPUT)


## Recomputed configuration

This summary is calculated before visualization and confirms the budget, group count, and DB index used by all four panels.


In [ ]:
metadata = artifacts['metadata']
summary = pd.DataFrame([{
    'visualized_profiles': int(artifacts['groups']['pool_images'].sum()),
    'gmm_groups': metadata['clusters'],
    'db_index': metadata['db_index'],
    'multiplication_factor': metadata['factor'],
    'alpha': metadata['alpha'],
    'candidates': int(artifacts['groups']['candidates'].sum()),
    'acquired': int(artifacts['groups']['acquired'].sum()),
}])
display(summary.style.format({'db_index': '{:.4f}', 'alpha': '{:.2f}'}))


## 1. DUA reference

DUA reduces each profile to the sum $U(i)$ and highlights the 20 most uncertain images in green. This panel shows the information that would be lost by reducing the class-wise profile to a scalar score before grouping.


In [ ]:
dua_figure = artifacts['interactive']['dua']
dua_figure.show(config={'scrollZoom': True, 'displaylogo': False})


## 2. Gaussian mixture model

The GMM receives the three profile components rather than their sum. Each color represents one group, and the dashed black wireframes represent two standard deviations of the Gaussian component. The lowest Davies–Bouldin index determines the number of groups.


In [ ]:
gmm_figure = artifacts['interactive']['gmm']
gmm_figure.show(config={'scrollZoom': True, 'displaylogo': False})


## 3. FM4 expansion and diversity-aware refinement

FM4 expands the analysis budget from 20 to 80 images. Sainte-Laguë determines each group quota, and the modular score selects candidates by combining uncertainty with distance from profiles already chosen. Dark red marks candidates; bright red identifies the 20 images also retained in the final batch.


In [ ]:
candidate_figure = artifacts['interactive']['candidates']
candidate_figure.show(config={'scrollZoom': True, 'displaylogo': False})


In [ ]:
group_summary = artifacts['groups'].rename(columns={
    'cluster': 'group',
    'pool_images': 'profiles_in_group',
    'candidates': 'candidates',
    'acquired': 'acquired',
    'quota': 'sainte_lague_quota',
})
display(group_summary.style.format({column: '{:.0f}' for column in group_summary.columns}))


## 4. Final CDU acquisition

After refinement, `retain_final_batch` orders the 80 candidates by $U(i)$ and retains exactly 20. Diversity structures the examined set; uncertainty determines which images enter the training manifest.


In [ ]:
final_figure = artifacts['interactive']['selected']
final_figure.show(config={'scrollZoom': True, 'displaylogo': False})


## Interpretation

A lower Davies–Bouldin index indicates groups that are more compact internally and better separated. Highly eccentric ellipsoids appear when one direction contains substantially more variance than the others.

The axes are class-wise detector uncertainty responses, not a projection of visual image content. The visualization explains how CDU retains class structure and how FM4, Sainte-Laguë, and Optα contribute to acquisition; the active-learning curves remain the quantitative performance evidence.
